In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import time

from pathlib import Path
from comet_ml import Experiment

import sys
import os
sys.path.append(os.path.abspath(".."))

from ampute import generate_missing_data
# from imputation import imputation, get_rmse
from experiment_runner import load_dataset, run_single_experiment, log_experiment

np.random.seed(42)

In [2]:
key = Path("../COMET_API_KEY.zshrc").read_text(encoding="utf-8").strip()
os.environ["COMET_API_KEY"] = key

experiment = Experiment(
    api_key=os.environ.get("COMET_API_KEY"),
    project_name="missing-data-imputation",
    workspace=None
)
experiment.set_name("default run (with mistake in knn and mice)")

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/kam2607kam/missing-data-imputation/8b6c74e91ca74c79ae8ac2b6576dcb88



In [3]:
import sklearn
print(sklearn.__version__)

1.8.0


In [3]:
# datasets = ["housing", "adult"]
datasets = ["adult"]

# колонки, в которых будем создавать пропуски
columns_target = {
    "housing": [{"median_income": "total_rooms"}, 
                {"total_rooms": "population"}, 
                {"housing_median_age": "households"}],
    "adult": [{"occupation": "hours-per-week"},
              {"workclass": "hours-per-week"}, 
              {"hours-per-week": "age"}],
}

#  глобальный таргет
global_target = {
    "housing": "median_house_value",
    "adult": "income",
}

# ratios = [0.05, 0.20, 0.50]
ratios = [0.20]

mechanisms = ["MCAR", "MAR"]

EXPERIMENT_CONFIG = {
    "Simple": [
        {"num_strategy": "mean"},
        {"num_strategy": "median"},
    ],
    # "KNN": [
    #     {"n_neighbors": 3},
    #     {"n_neighbors": 5},
    #     {"n_neighbors": 7},
    # ],
    "MICE": [
        {"max_iter": 10},
        {"max_iter": 50},
    ],
    "MissForest": [
        {"n_estimators": 100},
        {"n_estimators": 200},
    ]
}

for dataset_name in datasets:
    df = load_dataset(dataset_name)
    # удаляем глобальный таргет, чтобы случайно не подглядеть
    df = df.drop(columns=[global_target[dataset_name]])
    print(f"Dataset {dataset_name} loaded. Shape: {df.shape}")

    columns_with_missing = [list(d.keys())[0] for d in columns_target[dataset_name]]

    for mechanism in mechanisms:
        for ratio in ratios:

            df_incomplete, mask = generate_missing_data(
                data=df, 
                columns_config=columns_target[dataset_name],
                mechanism=mechanism, 
                ratio=ratio)
            
            print(f"Missing data generated for {dataset_name} | {mechanism} | Ratio: {ratio}")
            # print(f"counts missing: {mask.sum()} / {mask.size}")

            for method, param_list in EXPERIMENT_CONFIG.items():
                for params in param_list:
                    rmse, acc, time_taken = run_single_experiment(
                        df, 
                        df_incomplete, 
                        mask, 
                        method, 
                        params, 
                        columns_with_missing)
                    
                    params_str = "_".join([f"{k}={v}" for k, v in params.items()])

                    # model_accuracy = ... (метрика модели, если есть)

                    # Логируем параметры эксперимента
                    # log_experiment(
                    #     experiment,
                    #     dataset_name,
                    #     mechanism,
                    #     method,
                    #     params,
                    #     ratio,
                    #     rmse,
                    #     acc,
                    #     time_taken
                    # )

                    rmse_str = f"{rmse:.4f}" if rmse is not None else "N/A"
                    acc_str = f"{acc:.4f}" if acc is not None else "N/A"

                    print(f"Done: {dataset_name} | {mechanism} | {method} ({params_str}) | {ratio} | RMSE: {rmse_str} | ACC: {acc_str} | Time: {time_taken:.2f} sec")



Dataset adult loaded. Shape: (5000, 14)
Missing data generated for adult | MCAR | Ratio: 0.2
Done: adult | MCAR | Simple (num_strategy=mean) | 0.2 | RMSE: 1.0006 | ACC: 0.3955 | Time: 0.01 sec
Done: adult | MCAR | Simple (num_strategy=median) | 0.2 | RMSE: 1.0019 | ACC: 0.3955 | Time: 0.01 sec
Done: adult | MCAR | MICE (max_iter=10) | 0.2 | RMSE: 0.9193 | ACC: 0.5360 | Time: 2.75 sec
Done: adult | MCAR | MICE (max_iter=50) | 0.2 | RMSE: 0.9193 | ACC: 0.5360 | Time: 13.13 sec
Done: adult | MCAR | MissForest (n_estimators=100) | 0.2 | RMSE: 0.9013 | ACC: 0.4985 | Time: 5.31 sec
Done: adult | MCAR | MissForest (n_estimators=200) | 0.2 | RMSE: 0.9044 | ACC: 0.4945 | Time: 9.81 sec
Missing data generated for adult | MAR | Ratio: 0.2
Done: adult | MAR | Simple (num_strategy=mean) | 0.2 | RMSE: 1.0028 | ACC: 0.3980 | Time: 0.01 sec
Done: adult | MAR | Simple (num_strategy=median) | 0.2 | RMSE: 1.0042 | ACC: 0.3980 | Time: 0.01 sec
Done: adult | MAR | MICE (max_iter=10) | 0.2 | RMSE: 0.9448 | 

In [4]:
# experiment.log_table("final_results.csv", df_results)

experiment.end()

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : default run (with mistake in knn and mice)
COMET INFO:     url                   : https://www.comet.com/kam2607kam/missing-data-imputation/8b6c74e91ca74c79ae8ac2b6576dcb88
COMET INFO:   Metrics:
COMET INFO:     adult|mech=MAR|method=KNN|params=n_neighbors=3|ratio=0.05|metric=accuracy           : 0.27559377559377557
COMET INFO:     adult|mech=MAR|method=KNN|params=n_neighbors=3|ratio=0.05|metric=rmse               : 1.0374357626688926
COMET INFO:     adult|mech=MAR|method=KNN|params=n_neighbors=3|ratio=0.05|metric=time               : 15.086054874991532
COMET INFO:     adult|mech=MAR|method=KNN|params=n_neighbors=3|ratio=0.2|metric=accuracy          